# Подбор гиперпараметров.
## Использование Optuna

**Это вспомогательный ноутбук, он по сути копия основного**\
Подбирать гиперпараметры было удобнее в отдельном ноутбуке. Оставляю его чтобы вы не думали, что мы делали всё методом произвольного выбора.

In [1]:
import json
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from dotenv import load_dotenv
from catboost import CatBoostClassifier
from sqlalchemy import create_engine, text
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight

print('Необходимые библиотеки загружены.')

Необходимые библиотеки загружены.


In [2]:
# загрузка переменных из .env
load_dotenv()

# получаем токен из окружения
POSTGRES_CONNECTION_URL = os.getenv("POSTGRES_CONNECTION_URL")

try:
    # 1. Загрузка данных
    engine = create_engine(POSTGRES_CONNECTION_URL)

except Exception as e:
    print(f"An error occurred: {e}")

finally:
    # engine.dispose()
    print("Connected.")

Connected.


In [3]:
customer_features = pd.read_sql_query(
  f"""SELECT
  cft.customer_unique_id,
  cft.timeline_date,
  cft.city_size,
  cft.city_center_distance_km,
  cft.city_center_level,
  cft.days_after_order,
  cft.days_before_order,
  cft.is_customer_will_order,
  cft.days_before_order < 30 as is_customer_will_month,
-- -- cft.state_density,
  cft.state_income,
--  cft.state_north_south_index,
  cft.last_delivery_distance_km,
--  cft.last_delivery_delay_days,
  cft.last_delivery_cost,
  cft.last_order_cost,
-- -- cft.last_cancelled,
-- -- cft.last_delivered,
  cft.last_review_score,
  cft.last_review_length,
  cft.last_review_match_description,
  cft.last_review_product_state,
  cft.last_review_recommend,
  cft.last_review_sentiment,
  cft.last_review_buy_again,
  cft.last_review_want_leave,
  cft.last_review_unexpected_issues,
  cft.last_review_quality_happy,
  cft.avg_order_interval_days,
  coalesce(cft.main_payment_type, 'not_defined') as main_payment_type,
--  cft.last_order_month,
  cft.last_order_day_of_year,

--  srm.retention_rate_percent,
    srm.weighted_retention_rate,
--  srm.churn_rate_percent,
    srm.weighted_churn_rate,
--  srm.avg_review_score,
--  srm.weighted_avg_review_score,
-- --  srm.review_count,
  srm.percent_null_reviews,
  srm.dispersion_review_score,
  srm.avg_review_message_length,
--  srm.fake_review_probability,
  srm.product_count,
  srm.order_date_variance,

  p.product_name_lenght,
  p.product_description_lenght,
  p.product_photos_qty,
--  p.product_weight_g,
--  p.product_length_cm,
--  p.product_height_cm,
--  p.product_width_cm,
  --prm.weighted_retention_rate as product_weighted_retention_rate,
  --prm.weighted_churn_rate as product_weighted_churn_rate
  coalesce(abcxyz.category_adc, 'C') as category_adc,
  coalesce(abcxyz.category_xyz, 'Z') as category_xyz

FROM customer_features_timeline cft
left join orders o on o.order_id ::uuid=cft.order_id 
--full join orders_items oi on oi.order_id =o.order_id
inner join orders_items oi on oi.order_id =o.order_id
left join seller_retention_metrics srm on srm.seller_id =oi.seller_id
left join products p on p.product_id =oi.product_id
left join product_retention_metrics prm on prm.product_id =oi.product_id
left join public.analysis_abc_2_xyz abcxyz on abcxyz.product_category_name =p.product_category_name

where cft.timeline_date <>'2018-05-02' and cft.id is not null""", engine)


### Случайная генерация идентификаторов тестовых кастомеров

In [4]:
import numpy as np
customer_unique_ids = pd.read_sql_query(f"""SELECT distinct cft.customer_unique_id from customer_features_timeline cft""", engine)
np.random.seed(42)  # чтобы воспроизводилось
test_customer_ids = customer_unique_ids.sample(frac=0.1)['customer_unique_id'].tolist()

## Пробуем обучить модель

- Используем CatBoost для бинарной классификации.
- Разделяем данные по времени (train до 2018-02-01, test после).
- Учитываем дисбаланс классов (3% повторных заказов).

In [5]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_sample_weight
import matplotlib.pyplot as plt
import seaborn as sns


# Убедимся, что timeline_date — datetime
customer_features['timeline_date'] = pd.to_datetime(customer_features['timeline_date'])

# Выбираем дату-разделитель (например, 2025-01-01)
split_date = pd.to_datetime("2018-02-01")

train_df = customer_features[
    (~customer_features['customer_unique_id'].isin(test_customer_ids)) &
    (customer_features['timeline_date'] < split_date)
]

test_df = customer_features[
    (customer_features['customer_unique_id'].isin(test_customer_ids)) &
    (customer_features['timeline_date'] >= split_date)
]

# Убираем object/string и нечисловые данные
# Список категориальных признаков, которые НЕ нужно исключать
cat_features = ['category_adc', 'category_xyz', 'main_payment_type']
cols_to_exclude = [
    col for col in train_df.select_dtypes(include=['object', 'string']).columns
    if col not in cat_features
]
train_df = train_df.drop(columns=cols_to_exclude)
test_df = test_df.drop(columns=cols_to_exclude)

# X и y
drop_cols = ['is_customer_will_order', 'is_customer_will_month', 'days_before_order', 'timeline_date']
X_train = train_df.drop(columns=drop_cols, errors='ignore')
y_train = train_df['is_customer_will_order']

X_test = test_df.drop(columns=drop_cols, errors='ignore')
y_test = test_df['is_customer_will_order']

# NaN → среднее
X_train = X_train.fillna(X_train.mean(numeric_only=True))
X_test = X_test.fillna(X_train.mean(numeric_only=True))  # подгонка под train

# Веса классов
sample_weights = compute_sample_weight('balanced', y_train)

In [7]:
import optuna
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight

# Оборачиваем всё в одну функцию для Optuna
def objective(trial):
    # Подбираемые гиперпараметры
    params = {
        'iterations': 2000,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-2, 10.0, log=True),
        'random_strength': trial.suggest_float('random_strength', 1e-2, 10.0, log=True),
        'eval_metric': 'F1',
        'loss_function': 'Logloss',
        'verbose': 0,
        'random_seed': 42,
        'cat_features': cat_features,
        'early_stopping_rounds': 100
    }

    model = CatBoostClassifier(**params)
    sample_weights = compute_sample_weight('balanced', y_train)

    model.fit(X_train, y_train, sample_weight=sample_weights, eval_set=(X_test, y_test), use_best_model=True)
    y_pred = model.predict(X_test)
    return f1_score(y_test, y_pred)

# Запуск Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30)  # Можно изменить количество проб

# Лучшие параметры
print("Best trial:")
print(study.best_trial)



[I 2025-04-19 14:42:45,642] A new study created in memory with name: no-name-e38e8759-c0e4-4f41-aa7c-4a09a400f8d4
[I 2025-04-19 14:43:08,938] Trial 0 finished with value: 0.15568862275449102 and parameters: {'learning_rate': 0.052739015074485425, 'depth': 4, 'l2_leaf_reg': 0.22725306591482636, 'random_strength': 0.013466616642070096}. Best is trial 0 with value: 0.15568862275449102.
[I 2025-04-19 14:43:14,939] Trial 1 finished with value: 0.14418125643666324 and parameters: {'learning_rate': 0.014517139087560093, 'depth': 6, 'l2_leaf_reg': 4.436794250244416, 'random_strength': 1.0881503495794267}. Best is trial 0 with value: 0.15568862275449102.
[I 2025-04-19 14:43:41,357] Trial 2 finished with value: 0.1527864746399499 and parameters: {'learning_rate': 0.010416559258507409, 'depth': 10, 'l2_leaf_reg': 0.2747194001307745, 'random_strength': 2.526142849085164}. Best is trial 0 with value: 0.15568862275449102.
[I 2025-04-19 14:43:45,539] Trial 3 finished with value: 0.1371610845295056 an

Best trial:
FrozenTrial(number=26, state=TrialState.COMPLETE, values=[0.2644628099173554], datetime_start=datetime.datetime(2025, 4, 19, 14, 59, 3, 221730), datetime_complete=datetime.datetime(2025, 4, 19, 15, 1, 14, 526350), params={'learning_rate': 0.06777903832127262, 'depth': 10, 'l2_leaf_reg': 9.0490576545471, 'random_strength': 0.7961766953481294}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.1, log=True, low=0.005, step=None), 'depth': IntDistribution(high=10, log=False, low=4, step=1), 'l2_leaf_reg': FloatDistribution(high=10.0, log=True, low=0.01, step=None), 'random_strength': FloatDistribution(high=10.0, log=True, low=0.01, step=None)}, trial_id=26, value=None)


In [ ]:
# Переобучение с лучшими параметрами
best_params = study.best_params

best_model = CatBoostClassifier(**best_params)
best_model.fit(X_train, y_train, sample_weight=sample_weights, eval_set=(X_test, y_test), use_best_model=True)

# Предсказания и отчёт
print(classification_report(y_test, best_model.predict(X_test)))